# Seminário Modelagem Economica Baseada em Agentes - SCX5013

## 1. Visão geral e motivação do modelo

### 1.1 Trabalhos abordados

Iremos abordar o modelo proposto por  no trabalho 

- `Economic forecasting with an agent-based model`, Poledna et al. (2023)
-  `Behavioural agent-based economic forecasting in Julia`, implementação open source na linguagem Julia por Aldo et al. (2025)

Utilizaremos esta implementação em Julia para explorar o modelo de maneira prática. 

### 1.2 Motivadores do modelo

- Realizar previsões macroeconômicas utilizando modelagem baseada em agentes
- Construir uma modelagem verossimilhante, calibrando o modelo de acordo com as estatísticas governamentais e executando a simulação com escala 1:1 da população do país

O modelo inicial de foi calibrado para a economia da Áustria, que é um pais pequeno e altamente integrado com a economia da Europa ($ \approx 9M$ habitantes  e $\approx 50\%  $ do PIB em importações/exportações).

### 1.3 Inicialização do modelo

Com os comandos abaixo inicializamos as condições de partida do modelo, parâmetros (fixos) e variáveis de estado (evoluem no tempo).

In [6]:
import BeforeIT as Bit
import Random
using Plots, StatsPlots, DataFrames, Statistics, LinearAlgebra


parameters = Bit.AUSTRIA2010Q1.parameters;
initial_conditions = Bit.AUSTRIA2010Q1.initial_conditions;
model = Bit.Model(parameters,initial_conditions);

### 1.4 Agentes do modelo

Pomdemos inspecionar as condições iniciais para nos informarmos sobre os agentes do modelo.

Os agentes do modelo são:
- Trabalhadores ativos (empregados e desempregados)
- Trabalhadores inativos (aposentados)
- Empresas (e donos de empresas)
- Governos (central e locais)
- Banco comercial (agente representativo)
- Banco central
- Resto do mundo (agente representativo)


In [8]:
p = model.prop

agent_table = DataFrame(
        "Agent type"        => ["Trabalhadores ativos (empregados e desempregados)",
                                "Trabalhadores inativos",
                                "Empresas (e donos de empresas)",
                                "Governos (central e locais)",
                                "Consumidores externos",
                                "Empresas estrangeiras (importadoras)"],
        "Count"             => [p.H_act, p.H_inact, p.I, p.J, p.L, p.G],
    )

Row,Agent type,Count
,String,Int64
1,Trabalhadores ativos (empregados e desempregados),4743
2,Trabalhadores inativos,4130
3,Empresas (e donos de empresas),624
4,Governos (central e locais),156
5,Consumidores externos,312
6,Empresas estrangeiras (importadoras),62



Cada etapa temporal do modelo equivale a um trimestre. As etapas-chave que levam o modelo de $t$ para $t+1$ são:
1. Formação de expectativas coletivas (regra AR(1)) 
2. Execução dos choques exógenos 
3. Execução dos mercados de crédito e trabalho
4. Execução dos mercados de bens  
4. Atualização da contabilidade

Na implementação em Julia, isto se traduz em 47 funções executadas sequencialmente. A partir desta execução, podemos verificar como as estatísticas agregadas da economia evolvem ao longo do tempo:
- Crescimento da ecônomia
- Inflação
- Desemprego

Observamos também que o modelo possui propriedades emergentes que também são verificadas na ecônomia, como:
- Ciclos econômicos endógenos 
- Crescimento sustentado do PIB (países industrializados)


Abaixo, vamos veremos a progressão de algumas dessas varíaveis a partir de uma execução real do modelo. 

In [1]:
# TODO inserir código inicializando o modelo, verificar as variáveis iniciais e simular uma rodada

## 2. Regras autoregressivas de ordem 1 - AR(1)

Um aspecto central do modelo são as regras autoregressivas de ordem 1 e por isso iremos abordar o que são e como são utilizadas. 

As regras autoregressivas de ordem 1 são a estivamativa de uma variável no futuro, $z_{t+1}$, a partir de uma relação linear da variável no passo de tempo anterior, ou seja:

$z_{t+1} = \alpha z_{t} + \beta$

Em Poledna et. al, essa regra é utilizada de maneira otimizada para que os agentes possam construir suas expectativas em relação ao futuro econômico. Para isso, é feito um encaixe dos parâmetros $\alpha$ e $\beta$ para toda a série histórica da variável $z_{t}$ antes de aplicar a regra.

Esse procedimento é utilizado para calcular as `expectativas` das seguintes variáveis econômicas:
- Crescimento do PIB
- Inflação

O argumento principal é que esta é uma regra simples mas poderosa, onde os agentes ignoram as não linearidades que estão por trás das variáveis econômicas, e mesmo assim conseguem construir uma previsão conjunta da direção da econômia.  

Além disso, cada execução da regra é acompanhada por um choque exógeno proporcional ao quanto a regra autoregressva não consegue explicar da séria histórica da varíavel {TODO qual o nome?}. 

Adicionalmente, as seguintes variáveis também são modeladas como processos AR(1) baseados em dados históricos {TODO verificar se há otimização de parâmetros }:
- Importações
- Exportações
- Inflação da área do euro
- Crescimento da área do euro

In [ ]:
# TODO código da regra autoregressiva

## 3. Calibração para as contas nacionais 

O trabalho busca se adequar das estatísticas calculadas pelos órgãos de contabilidade nacional. Nesse sentido, os parâmetros do modelo, que são muitos, em grande medida são retirados de diretamente dos relatórios destes órgãos ou calculados a partir deles, o que torna o modelo mais verossimilhante e reduz a necessidade de técnicas de estimação de parâmetros. 

Abaixo iremos explorar alguns dos parâmetros mais interessantes do modelo.

A matrix de `input-output`, concebida por Leontief, descreve as relações entre os diferentes setores produtivos. Cada linha representa o setor $s$ e cada coluna o produto $g$. A entrada da matriz representa o quanto do setor $s$ é utilizado para produzir o bem $g$.  

Outro ponto importante é a consistência estoque-fluxo do modelo. Isso quer dizer que não há um dinheiro ou recurso entrando de maneira "estranha" no modelo, ou seja, sabemos para cada transação, quem quem gastou, quem recebeu e como isso reflete nas contas de cada uma das partes. Na prática, isso fornece ao modelo a consistência contábil que é utilizada na mensuração dos indicadores macroeconomicos. 

In [ ]:
# TODO codigos da matrix input output e 

## 4. Mercados com fricção  ("Search and Matching")

Outra característica relevante do modelo é como os agentes econômicos, empresas, famílias e governos, se relacionam em mercados para produzir e consumir. 
 
Isto se dá através de mercados descentralizados, caracterizados pelo mecanismo de "search and matching", onde os consumidores são ordenados em ordem aleatória e as empresas são escolhidas com probabilidade proporcional ao seu tamanho e preço praticado. 

De acordo com os autores, este método permite reproduzir frições nas relações de troca e portanto mercados que não estão necessariamente em equilíbrio. Assim, é possível que consumidores não consigam comprar tudo que tinham em seu orçamento e é possível que empresas não consigam vender tudo aquilo que produziram. 

No entanto, caso não hajam mudanças abruptas no sistema, as empresas e consumidores tendem a caminhar em um percurso próximo ao equilíbrio onde a oferta se iguala ou está próxima do equilíbrio.      

In [ ]:
# TODO codigo search and matching 

## 5. Resultados
- Mensuração dos resultados (RMSE)

## 6. Críticas e alterações propostas  

- Sistema bancário
- "Autorealizável"

## TODOs

- Verificar quanto é um dinheiro na simulação